Purpose

Given electricity prices for each of the 48 hours, find the best schedule of one generator: when it is on, and how much it produces. "Best" means lowest cost minus what the prices pay it.

In [17]:
import gurobipy as gp 
from gurobipy import GRB

import numpy as np 

In [18]:
import sys
from pathlib import Path


# make uc/data.py importable
ROOT = Path.cwd()
while not (ROOT / "uc").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from uc.data import load_generators

gen = load_generators(ROOT / "data" / "generators" / "out" / "10_0_1_w.csv")

# parameters of the 1 generator
g = 0                          # which generator (0 to 9)
min_out  = gen["min_out"][g]   # minimum output when on (MW)
max_out  = gen["max_out"][g]   # maximum output (MW)
nol_cost = gen["nol_cost"][g]  # cost per hour just for being on
mar_cost = gen["mar_cost"][g]  # cost per MW produced


In [19]:
#inputs for 1 generator 
T=48
hours=range(T)

#variables coming from the data NEED TO CHANGE LATER 
y_d = np.full(T, 60.0)        
y_r = np.zeros(T)


m = gp.Model("pricing")
m.setParam("OutputFlag", 0)

In [20]:

#new variables 
u = m.addVars(hours, vtype=GRB.BINARY, name="u")
p = m.addVars(hours, lb= 0.0, ub=max_out, vtype=GRB.CONTINUOUS, name="p")

#constraint: output limits
m.addConstrs((min_out*u[t] <= p[t] for t in hours), name="lo")
m.addConstrs((p[t] <= max_out* u[t] for t in hours), name ="hi")


#priced objective:
obj = gp.quicksum(nol_cost*u[t] + mar_cost * p[t] - y_d[t] *p[t] - y_r[t] * (max_out *u[t] - p[t]) for t in hours)
m.setObjective(obj, GRB.MINIMIZE)
m.optimize()



In [21]:
#print results 
print("value:", m.ObjVal)
print("on:", [int(round(u[t].X)) for t in hours])
print("p:", [round(p[t].X, 2) for t in hours])

value: 0.0
on: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
p: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


Making the hours dependent, adding starting cost. Adding variables for when the plant was off at hour t-1 and is on at hour t, as well as if the plant was on at hour t-1 and is off at t 

In [22]:
#creating new variables
v = m.addVars(hours, vtype=GRB.BINARY, name="v") 
w = m.addVars(hours, vtype=GRB.BINARY, name="w") 

#adding starting cost 
st_cost = gen["st_cost"][g]

#plant is on before hour 0 for testing 
u_init = 1  


In [23]:
#adding the constraints 
m.addConstrs((u[t] - u[t-1] == v[t] - w[t] for t in range(1, T)))
m.addConstrs((v[t] + w[t] <= 1 for t in hours))
#hour 0 needs different handling 
m.addConstr(u[0] - 1 == v[0] - w[0])

<gurobi.Constr *Awaiting Model Update*>

In [24]:
#new objective with additional costs 
obj1 = gp.quicksum(nol_cost*u[t] + mar_cost * p[t] + st_cost * v[t] - y_d[t] *p[t] - y_r[t] * (max_out *u[t] - p[t]) for t in hours)
m.setObjective(obj1, GRB.MINIMIZE)
m.optimize()

In [25]:
#print results 
print("v:", [int(round(v[t].X)) for t in hours])
print("w:", [int(round(w[t].X)) for t in hours])


v: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
w: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
#tests with Claude to make sure it works 
print("generator:", g, " st_cost:", round(st_cost, 2))

HIGH = 1000.0   # "very high" price

def run_test(name, y_d, y_r=np.zeros(T)):
    obj1 = gp.quicksum(nol_cost*u[t] + mar_cost * p[t] + st_cost * v[t] - y_d[t] *p[t] - y_r[t] * (max_out *u[t] - p[t]) for t in hours)
    m.setObjective(obj1, GRB.MINIMIZE)
    m.optimize()
    print("\n" + name)
    print("value:", round(m.ObjVal, 2))
    print("on:", [t for t in hours if u[t].X > 0.5])
    print("v = 1 at hours:", [t for t in hours if v[t].X > 0.5])
    print("w = 1 at hours:", [t for t in hours if w[t].X > 0.5])

#6a per-hour value when on at full output
def hour_value(price):
    return nol_cost + (mar_cost - price) * max_out

#test 1: all prices 0
run_test("TEST 1: all prices 0", np.zeros(T))
print("expected: on none, w[0] = 1, value 0")

#test 2: very high price in hours 0-9
y = np.zeros(T); y[0:10] = HIGH
run_test("TEST 2: high price in hours 0-9", y)
print("expected: on 0-9, no v, w[10] = 1, value", round(10 * hour_value(HIGH), 2))

#test 3: very high price in hours 20-29
y = np.zeros(T); y[20:30] = HIGH
run_test("TEST 3: high price in hours 20-29", y)
print("expected: w[0] = 1, v[20] = 1, on 20-29, w[30] = 1, value", round(10 * hour_value(HIGH) + st_cost, 2))

#test 4: price in hours 20-29 slightly above mar_cost, 10-hour profit < st_cost
price4 = mar_cost + (nol_cost + st_cost / 20) / max_out
y = np.zeros(T); y[20:30] = price4
run_test(f"TEST 4: price {price4:.2f} in hours 20-29", y)
print("10-hour profit if on:", round(-10 * hour_value(price4), 2), "< st_cost", round(st_cost, 2))
print("expected: on none, w[0] = 1, value 0")

#6a re-run: high price every hour
run_test("6a: high price every hour", np.full(T, HIGH))
print("expected: on all hours, no v, value", round(T * hour_value(HIGH), 2))


generator: 0  st_cost: 2074.35

TEST 1: all prices 0
value: 0.0
on: []
v = 1 at hours: []
w = 1 at hours: [0]
expected: on none, w[0] = 1, value 0

TEST 2: high price in hours 0-9
value: -1104961.9
on: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
v = 1 at hours: []
w = 1 at hours: [10]
expected: on 0-9, no v, w[10] = 1, value -1104961.9

TEST 3: high price in hours 20-29
value: -1102887.55
on: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
v = 1 at hours: [20]
w = 1 at hours: [0, 30]
expected: w[0] = 1, v[20] = 1, on 20-29, w[30] = 1, value -1102887.55

TEST 4: price 64.47 in hours 20-29
value: 0.0
on: []
v = 1 at hours: []
w = 1 at hours: [0]
10-hour profit if on: 1037.17 < st_cost 2074.35
expected: on none, w[0] = 1, value 0

6a: high price every hour
value: -5303817.12
on: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]
v = 1 at hours: []
w = 1 at hours: []
expected: on

Adding new minimum up and down times that weren't there before 

In [27]:
#minimum up and minimum down 
min_up = gen["min_up"][g]
min_up = int(min_up)
min_down = gen["min_down"][g]
min_down = int(min_down)

In [ ]:
#adding new constraints 
m.addConstrs(gp.quicksum(v[i]for i in range(max(0, t-min_up +1), t)) <= u[t] for t in hours)
m.addConstrs(gp.quicksum(w[i]for i in range(max(0, t-min_down + 1), t)) <= 1-u[t] for t in hours)


{0: <gurobi.Constr *Awaiting Model Update*>,
 1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>,
 7: <gurobi.Constr *Awaiting Model Update*>,
 8: <gurobi.Constr *Awaiting Model Update*>,
 9: <gurobi.Constr *Awaiting Model Update*>,
 10: <gurobi.Constr *Awaiting Model Update*>,
 11: <gurobi.Constr *Awaiting Model Update*>,
 12: <gurobi.Constr *Awaiting Model Update*>,
 13: <gurobi.Constr *Awaiting Model Update*>,
 14: <gurobi.Constr *Awaiting Model Update*>,
 15: <gurobi.Constr *Awaiting Model Update*>,
 16: <gurobi.Constr *Awaiting Model Update*>,
 17: <gurobi.Constr *Awaiting Model Update*>,
 18: <gurobi.Constr *Awaiting Model Update*>,
 19: <gurobi.Constr *Awaiting Model Update*>,
 20: <gurobi.Constr *Awaiting Model Update*>,
 21: <gurobi.Constr *Awaiting Model Update*>

In [30]:
#show results 

obj1 = gp.quicksum(nol_cost*u[t] + mar_cost * p[t] + st_cost * v[t] - y_d[t] *p[t] - y_r[t] * (max_out *u[t] - p[t]) for t in hours)
m.setObjective(obj1, GRB.MINIMIZE)
m.optimize()

#print results 
print("v:", [int(round(v[t].X)) for t in hours])
print("w:", [int(round(w[t].X)) for t in hours])

v: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
w: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
#tests for minimum up / down times made with Claude to check it works 
print("generator:", g, " min_up:", min_up, " min_down:", min_down, " st_cost:", round(st_cost, 2))

HIGH = 1000.0   # "very high" price

def run_test2(name, y_d, y_r=np.zeros(T)):
    obj1 = gp.quicksum(nol_cost*u[t] + mar_cost * p[t] + st_cost * v[t] - y_d[t] *p[t] - y_r[t] * (max_out *u[t] - p[t]) for t in hours)
    m.setObjective(obj1, GRB.MINIMIZE)
    m.optimize()
    print("\n" + name)
    if m.Status != GRB.OPTIMAL:
        print("NOT OPTIMAL, status:", m.Status)
        return
    print("value:", round(m.ObjVal, 2))
    print("on:", [t for t in hours if u[t].X > 0.5])
    print("v = 1 at hours:", [t for t in hours if v[t].X > 0.5])
    print("w = 1 at hours:", [t for t in hours if w[t].X > 0.5])
    print("p (hours on):", {t: round(p[t].X, 1) for t in hours if u[t].X > 0.5})

#cost of one hour on, given price, when it produces q MW
def hour_cost(price, q):
    return nol_cost + (mar_cost - price) * q

#test 1: one-hour spike at hour 20
y = np.zeros(T); y[20] = HIGH
run_test2("TEST 1: spike at hour 20", y)
hand1 = hour_cost(HIGH, max_out) + (min_up - 1) * hour_cost(0, min_out) + st_cost
print(f"expected: one start, on {min_up} hours in a row including hour 20 (e.g. 20-{20 + min_up - 1}), extra hours at min_out, value {hand1:.2f}")

#test 2: high price everywhere except hour 20 (price 0)
y = np.full(T, HIGH); y[20] = 0
run_test2("TEST 2: cheap hour 20", y)
hand2 = (T - 1) * hour_cost(HIGH, max_out) + hour_cost(0, min_out)
print(f"expected: on all hours, p[20] = min_out = {min_out}, no v, value {hand2:.2f}")

#test 3: spikes at hours 10 and 12
y = np.zeros(T); y[10] = HIGH; y[12] = HIGH
run_test2("TEST 3: spikes at hours 10 and 12", y)
last = max(12, 10 + min_up - 1)
hand3 = 2 * hour_cost(HIGH, max_out) + (last - 10 + 1 - 2) * hour_cost(0, min_out) + st_cost
print(f"expected: one start, on {last - 10 + 1} hours in a row covering 10-12 (no stop at 11), value {hand3:.2f}")

#re-run 6a / 6b tests
run_test2("6a: high price every hour", np.full(T, HIGH))
print("expected: on all hours, no v, value", round(T * hour_cost(HIGH, max_out), 2))
run_test2("6b TEST 1: all prices 0", np.zeros(T))
print("expected: on none, w[0] = 1, value 0")
y = np.zeros(T); y[0:10] = HIGH
run_test2("6b TEST 2: high price in hours 0-9", y)
print("expected: on 0-9, w[10] = 1, value", round(10 * hour_cost(HIGH, max_out), 2))
y = np.zeros(T); y[20:30] = HIGH
run_test2("6b TEST 3: high price in hours 20-29", y)
print("expected: v[20] = 1, on 20-29, w[30] = 1, value", round(10 * hour_cost(HIGH, max_out) + st_cost, 2))



generator: 0  min_up: 4  min_down: 3  st_cost: 2074.35

TEST 1: spike at hour 20
value: -100027.7
on: [20, 21, 22, 23]
v = 1 at hours: [20]
w = 1 at hours: [0, 24]
p (hours on): {20: 118.0, 21: 32.0, 22: 32.0, 23: 32.0}
expected: one start, on 4 hours in a row including hour 20 (e.g. 20-23), extra hours at min_out, value -100027.70

TEST 2: cheap hour 20
value: -5190522.88
on: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]
v = 1 at hours: []
w = 1 at hours: []
p (hours on): {0: 118.0, 1: 118.0, 2: 118.0, 3: 118.0, 4: 118.0, 5: 118.0, 6: 118.0, 7: 118.0, 8: 118.0, 9: 118.0, 10: 118.0, 11: 118.0, 12: 118.0, 13: 118.0, 14: 118.0, 15: 118.0, 16: 118.0, 17: 118.0, 18: 118.0, 19: 118.0, 20: 32.0, 21: 118.0, 22: 118.0, 23: 118.0, 24: 118.0, 25: 118.0, 26: 118.0, 27: 118.0, 28: 118.0, 29: 118.0, 30: 118.0, 31: 118.0, 32: 118.0, 33: 118.0, 34: 118.0, 35: 118.0,

Adding ramp limits: the plant can't change its output too fast from one hour to the next 

In [32]:
#new variables 

op_rampup = gen["op_rampup"][g]
st_rampup = gen["st_rampup"][g]
sh_rampdown = gen["sh_rampdown"][g]
op_ranmpdown = gen["op_rampdown"][g]

In [ ]:
#add new constraints 
m.addConstrs(p[t] - p[t-1] <= op_rampup * u[t-1] + st_rampup *v[t] for t in range(1, T))
m.addConstrs(p[t-1] - p[t] <= op_ramnpdown * u[t] + sh_rampdown *w[t] for t in range(1, T))


NameError: name 'op_rampdown' is not defined

In [34]:
obj1 = gp.quicksum(nol_cost*u[t] + mar_cost * p[t] + st_cost * v[t] - y_d[t] *p[t] - y_r[t] * (max_out *u[t] - p[t]) for t in hours)
m.setObjective(obj1, GRB.MINIMIZE)
m.optimize()



In [35]:
#tests for ramp limits
ru, rd = gen["op_rampup"][g], gen["op_rampdown"][g]
sru, srd = gen["st_rampup"][g], gen["sh_rampdown"][g]
print("generator:", g, " min_out:", min_out, " max_out:", max_out)
print(f"op_rampup: {ru:.2f}  op_rampdown: {rd:.2f}  st_rampup: {sru:.2f}  sh_rampdown: {srd:.2f}")
print("op_rampup < max_out - min_out:", ru < max_out - min_out)

HIGH = 1000.0   # "very high" price
tol = 1e-6

def run_test3(name, y_d, y_r=np.zeros(T)):
    obj1 = gp.quicksum(nol_cost*u[t] + mar_cost * p[t] + st_cost * v[t] - y_d[t] *p[t] - y_r[t] * (max_out *u[t] - p[t]) for t in hours)
    m.setObjective(obj1, GRB.MINIMIZE)
    m.optimize()
    print("\n" + name)
    if m.Status != GRB.OPTIMAL:
        print("NOT OPTIMAL, status:", m.Status)
        return
    P = [p[t].X for t in hours]
    U = [round(u[t].X) for t in hours]
    print("value:", round(m.ObjVal, 2))
    print("v = 1 at hours:", [t for t in hours if v[t].X > 0.5])
    print("w = 1 at hours:", [t for t in hours if w[t].X > 0.5])
    print("p:   ", [round(x, 1) for x in P])
    print("diff:", [round(P[t] - P[t-1], 1) for t in range(1, T)])
    #check every ramp rule
    ok = True
    for t in range(1, T):
        up, down = P[t] - P[t-1], P[t-1] - P[t]
        if U[t-1] == 0 and U[t] == 1 and P[t] > sru + tol:
            print(f"  FAIL start hour {t}: p = {P[t]:.2f} > st_rampup"); ok = False
        if U[t-1] == 1 and U[t] == 1 and up > ru + tol:
            print(f"  FAIL hour {t}: rise {up:.2f} > op_rampup"); ok = False
        if U[t-1] == 1 and U[t] == 1 and down > rd + tol:
            print(f"  FAIL hour {t}: fall {down:.2f} > op_rampdown"); ok = False
        if U[t-1] == 1 and U[t] == 0 and P[t-1] > srd + tol:
            print(f"  FAIL last on hour {t-1}: p = {P[t-1]:.2f} > sh_rampdown"); ok = False
    print("ramp checks:", "PASS" if ok else "FAIL")

#test 1: slow climb, zero price in hours 0-9, high from hour 10
y = np.zeros(T); y[10:] = HIGH
run_test3("TEST 1: slow climb", y)
print(f"expected: w[0] = 1, start at hour >= {min_down}, p at start <= {sru:.2f}, then rises <= {ru:.2f} per hour up to {max_out}")

#test 2: slow descent, high price in hours 0-29, then zero
y = np.zeros(T); y[0:30] = HIGH
run_test3("TEST 2: slow descent", y)
print(f"expected: falls <= {rd:.2f} per hour, last on hour p <= {srd:.2f}, may start dropping before hour 30")

#re-run earlier tests (6a, 6b, 6c)
earlier = {}
y = np.full(T, HIGH);                     earlier["6a: high every hour"] = y
y = np.zeros(T);                          earlier["6b-1: all prices 0"] = y
y = np.zeros(T); y[0:10] = HIGH;          earlier["6b-2: high in 0-9"] = y
y = np.zeros(T); y[20:30] = HIGH;         earlier["6b-3: high in 20-29"] = y
y = np.zeros(T); y[20] = HIGH;            earlier["6c-1: spike at 20"] = y
y = np.full(T, HIGH); y[20] = 0;          earlier["6c-2: cheap hour 20"] = y
y = np.zeros(T); y[10] = HIGH; y[12] = HIGH; earlier["6c-3: spikes at 10 and 12"] = y
for name, y in earlier.items():
    run_test3("RE-RUN " + name, y)


generator: 0  min_out: 32.0  max_out: 118.0
op_rampup: 42.00  op_rampdown: 32.60  st_rampup: 42.00  sh_rampdown: 32.60
op_rampup < max_out - min_out: True

TEST 1: slow climb
value: -4188668.36
v = 1 at hours: [8]
w = 1 at hours: [0]
p:    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 34.0, 76.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0, 118.0]
diff: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 34.0, 42.0, 42.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
ramp checks: PASS
expected: w[0] = 1, start at hour >= 3, p at start <= 42.00, then rises <= 42.00 per hour up to 118.0

TEST 2: slow descent
value: -3305608.6
v = 1 at hours: []
w = 1 at hours: [